# Business Overview

Olist is a Brazilian e-commerce platform that enables small and medium-sized sellers to sell their products across multiple online marketplaces through a single contract.

In addition to connecting sellers with customers, Olist coordinates order processing, logistics, and delivery to the final customer.

### Business Model

Olist generates revenue through three main streams:

- **Commission Fee:** A percentage of the value of successful sales.
- **Subscription Fee:** A fixed monthly fee charged to sellers.
- **Logistics Fee:** Fees related to transportation and logistics services.

### Business Context

Olist's success depends on both **sales growth** and **customer experience**. Therefore, this project focuses on three key areas:

1. **Logistics Performance:** Are orders delivered on time?
2. **Customer Satisfaction:** What factors are associated with lower customer ratings?
3. **Seller Performance:** Is the business dependent on a small number of sellers?

> **Data Scope:** The dataset covers the Brazilian market from **2016 to 2018**. Therefore, the analysis represents Olist's historical business performance during this period and should not be interpreted as its current state.

# Data Underastandig

In [4]:
# import libs
import pandas as pd
import numpy as np

In [5]:
# import dataset

df_orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
df_order_items = pd.read_csv('../data/raw/olist_order_items_dataset.csv')
df_customers = pd.read_csv('../data/raw/olist_customers_dataset.csv')
df_payments = pd.read_csv('../data/raw/olist_order_payments_dataset.csv')
df_products = pd.read_csv('../data/raw/olist_products_dataset.csv')
df_translation = pd.read_csv('../data/raw/product_category_name_translation.csv')
df_geolocation = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
df_reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')
df_sellers = pd.read_csv('../data/raw/olist_sellers_dataset.csv')

## orders table

In [47]:
df_orders.sample(n=5)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
49844,0de88914980b96be31b33a407f19ccfb,2397dd19bf74da4cd1fe4b5839199d91,delivered,2017-12-11 08:45:05,2017-12-11 08:52:27,2017-12-11 19:32:17,2017-12-20 20:39:14,2018-01-09 00:00:00
21959,2a11e293888cfc616767f5be01e7e5d2,a8369395689cc966efea14864a417708,delivered,2017-10-13 18:46:27,2017-10-14 18:34:41,2017-10-17 21:32:51,2017-10-26 19:46:43,2017-10-31 00:00:00
56687,1f6588173be9bf635ac895276a116522,94e5dec6a70d4203e76e8dfbd35706ac,shipped,2018-02-25 15:58:54,2018-02-25 16:10:23,2018-03-01 23:32:45,NaN,2018-03-19 00:00:00
51797,e571dfc21abcbe1ae8795d4a8a6b057a,c64aa85df0994669e681545091d74b64,delivered,2017-05-18 09:53:17,2017-05-18 10:05:16,2017-05-23 12:36:44,2017-05-29 16:04:05,2017-06-22 00:00:00
18999,4f287ea2c7dbdf8e22d05ec70cc1b9ab,c96df6d3deff85d63641900cc83bd2e4,delivered,2018-01-14 16:07:31,2018-01-14 16:31:06,2018-01-15 16:38:56,2018-01-29 18:26:40,2018-02-15 00:00:00


In [7]:
df_orders.info()

print("\ncolumns 5 to 7 are date but thire Dtype is Object. should be change in EDA\n")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB

columns 5 to 7 are date but thire Dtype is Object. should be change in EDA



In [8]:
print(df_orders.isnull().sum())

print("\n\ncolumns\norder_approved_at\n" \
      "order_delivered_carrier_date\n" \
      "order_delivered_customer_date\n" \
      "have missing value, Missing values may represent orders that have not yet reached that stage")

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


columns
order_approved_at
order_delivered_carrier_date
order_delivered_customer_date
have missing value, Missing values may represent orders that have not yet reached that stage


In [9]:
# each order is registered only once
orders_dup = df_orders.order_id.duplicated().sum()
print(f"\nnum of orders_id duplicated: {orders_dup}\n\
    order id don't have null or duplicated value and based on a dataset logical is Primary Key")

# # This customer ID is not duplicated because it creates a new customer ID with each order
customer_dup = df_orders.customer_id.duplicated().sum()
print(f"\nnum of customer id duplicated: {customer_dup}\n")

order_counts = df_orders['order_status'].value_counts()
print(order_counts)



num of orders_id duplicated: 0
    order id don't have null or duplicated value and based on a dataset logical is Primary Key

num of customer id duplicated: 0

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64


## order_items table

In [10]:
df_order_items.sample(5)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
41960,5f93c15063313abf1a114e682d6b6d71,1,51f876eb62be778c757503cf7f8a3b74,f84a00e60c73a49e7e851c9bdca3a5bb,2018-05-15 11:14:15,9.80,7.55
97561,dd50addabe54a9c924c82e73c4daac41,1,e24f73b7631ee3fbb2ab700a9acaa258,0cbcee27c791afa0cdcb08587a2013a8,2018-05-04 02:30:46,115.00,23.39
23042,34a1e7e0bd6987d552a7b050ad68a314,1,3f2146831fb32311f584fd931a0d51ad,725c32fa80c2faacc4fc88450d27314e,2017-05-22 03:15:34,159.99,9.93
63893,91eefa3cd52e8361bd4f6106356c6d0c,1,410d5b211dce5611e7a6412bf0cc30fa,2138ccb85b11a4ec1e37afbd1c8eda1f,2017-04-14 13:25:07,18.99,14.11
77185,afc091f97bfa9985b6ab413ad1a9891b,1,9e5f752e9e19cd6887063f004ee0da55,71593c7413973a1e160057b80d4958f6,2018-08-24 10:55:27,59.90,9.17


In [11]:
df_order_items.info()

print("\ncolumn shipping_limit_date is date type but  Dtype is Object")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB

column shipping_limit_date is date type but  Dtype is Object


In [12]:
# don't exist null values in order items
print(df_order_items.isnull().sum())

order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64


In [13]:
df_order_items[['freight_value', 'price']].agg(['min', 'max'])

# Some order items have a freight_value of 0.0. This should be 
# investigated during the EDA phase to determine whether
# it represents free shipping, promotional offers, or another rule ?

,freight_value,price
min,0.00,0.85
max,409.68,6735.00


## customers table

In [14]:
df_customers.sample(5)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
46787,bbd19dc1cbec6e7a4de9dd1ea15e2691,2953379eb7a88dc2f3ad11478e069174,41603,salvador,BA
26082,17bde6e1b8489fa8ee69d2606ef4abb5,ae37f0d2cc61ec9ec8726270abcbfbd9,15015,sao jose do rio preto,SP
83736,0a2a3b9d1077f4b509232f38c3f2c199,9a0eefac2d296d66e1ab1330e9632f14,95150,nova petropolis,RS
57439,26c7ac168e1433912a51b924fbd34d34,ef0996a1a279c26e7ecbd737be23d235,2290,sao paulo,SP
11311,9953831663e9cea0823e292888b62b4e,bd86b1862fe796149a4d2af57bd85451,48668,barra do tarrachil,BA


In [15]:
df_customers.info()

print("\ncount of null values")
print(df_customers.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_id               99441 non-null  object
 1   customer_unique_id        99441 non-null  object
 2   customer_zip_code_prefix  99441 non-null  int64 
 3   customer_city             99441 non-null  object
 4   customer_state            99441 non-null  object
dtypes: int64(1), object(4)
memory usage: 3.8+ MB

count of null values
customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64


In [16]:
df_customers.describe(include='object')

,customer_id,customer_unique_id,customer_city,customer_state
count,99441,99441,99441,99441
unique,99441,96096,4119,27
top,06b8999e2fba1a1fbc88172c00ba8bc7,8d50f5eadf50201ccdcedfb9e2ac8455,sao paulo,SP
freq,1,17,15540,41746


In [17]:
customer_unique_id_dup = df_customers['customer_unique_id'].duplicated().sum()
customer_id_dup = df_customers['customer_id'].duplicated().sum()

print(f'customer_unique_id_dup: {customer_unique_id_dup}')
print(f'customer_id_dup: {customer_id_dup}')

#  customer_id column contains no duplicate values
#  customer_unique_id column appears to identify repeat customers

customer_unique_id_dup: 3345
customer_id_dup: 0


#### each row represents a customer record associated with an order
- customer_id appears to be unique for each customer record (Primary Key)
- customer_unique_id may represent the actual customer across multiple orders
- No missing values were found


## products table

In [18]:
df_products.sample(5)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
23745,42737e3759311804e830f222c6135e30,eletronicos,56.0,431.0,1.0,5950.0,98.0,17.0,18.0
8911,711ffb7574aa22c89aa744dd74fefd5c,ferramentas_jardim,63.0,1330.0,4.0,3100.0,37.0,9.0,16.0
26658,3fe46d73f7527ad59e36c21c3e783041,consoles_games,58.0,1099.0,1.0,900.0,30.0,30.0,40.0
22206,e8cb961b90ee6176d66183e712e0dcfb,cama_mesa_banho,35.0,221.0,2.0,1300.0,38.0,8.0,28.0
25033,c98f9b6939bc77914b077d6aa93cd97c,dvds_blu_ray,59.0,1085.0,1.0,200.0,20.0,2.0,14.0


In [19]:
print(f"shape of products table: {df_products.shape}\n\n")

df_products.info()

print("\n\ncount of null values")
df_products.isnull().sum()

# the product may not have complete information yet

shape of products table: (32951, 9)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB


count of null values


product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

In [20]:
df_products.duplicated(subset=['product_id']).sum()
# product_id appears to be the Primary Key

np.int64(0)

In [21]:
df_products.describe(include='object')

,product_id,product_category_name
count,32951,32341
unique,32951,73
top,1e9e8ef04dbcff4541ed26657ea517e5,cama_mesa_banho
freq,1,3029


In [22]:
df_products[['product_name_lenght','product_description_lenght',
             'product_photos_qty','product_weight_g',
             'product_length_cm','product_height_cm',
             'product_width_cm']].agg(['min', 'max'])

,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
min,5.0,4.0,1.0,0.0,7.0,2.0,6.0
max,76.0,3992.0,20.0,40425.0,105.0,105.0,118.0


### Initial Observations
- the table contains product attributes
- each row represents a unique product
- table does not contain any foreign keys
- no duplicate values were found in the product_id column
- Several product attributes contain missing values
- Physical dimensions are available for shipping analysis

## sellers table

In [23]:
df_sellers.sample(5)

,seller_id,seller_zip_code_prefix,seller_city,seller_state
1964,8629c241b3622ac12cd4118afdc8d394,14090,ribeirao preto,SP
2220,db2956745b3a8e9f3785c99f34b5d25e,11600,sao sebastiao,SP
1719,929f342384a6607afe143d789ade1316,86065,londrina,PR
2869,3febca52652e7209509ccfe61cbde40e,8596,itaquaquecetuba,SP
1259,f114dca2828bf718548db175ebe2cfcd,86192,cambe,PR


In [24]:
print(f"shape of sellers table: {df_sellers.shape}\n\n")

df_sellers.info()

print("\n\ncount of null values")
df_sellers.isnull().sum()

shape of sellers table: (3095, 4)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   int64 
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   object
dtypes: int64(1), object(3)
memory usage: 96.8+ KB


count of null values


seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

In [25]:
df_sellers.describe(include='object')
# seller_id appears to be the Primary Key (don't hve duplicate seller)

,seller_id,seller_city,seller_state
count,3095,3095,3095
unique,3095,611,23
top,3442f8959a84dea7ee197c632cb2df15,sao paulo,SP
freq,1,694,1849


### Initial Observations
- the table contains seller-related information
- each row represents a unique seller
- the table does not contain any foreign keys
- no duplicate values were found in the seller_id column
- no missing values were found
- seller location information is available for geographic and logistics analysis

## geolocation table

In [26]:
df_geolocation.sample(5)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
107176,4807,-23.711153,-46.701002,sao paulo,SP
261543,11790,-24.277678,-47.229404,pedro de toledo,SP
421261,21051,-22.875885,-43.269584,rio de janeiro,RJ
800447,74705,-16.672028,-49.217627,goiânia,GO
84733,4222,-23.596577,-46.590797,sao paulo,SP


In [27]:
print(f"shape of geolocation table: {df_geolocation.shape}\n\n")

df_geolocation.info()

print("\n\ncount of null values")
df_geolocation.isnull().sum()

shape of geolocation table: (1000163, 5)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000163 entries, 0 to 1000162
Data columns (total 5 columns):
 #   Column                       Non-Null Count    Dtype  
---  ------                       --------------    -----  
 0   geolocation_zip_code_prefix  1000163 non-null  int64  
 1   geolocation_lat              1000163 non-null  float64
 2   geolocation_lng              1000163 non-null  float64
 3   geolocation_city             1000163 non-null  object 
 4   geolocation_state            1000163 non-null  object 
dtypes: float64(2), int64(1), object(2)
memory usage: 38.2+ MB


count of null values


geolocation_zip_code_prefix    0
geolocation_lat                0
geolocation_lng                0
geolocation_city               0
geolocation_state              0
dtype: int64

In [28]:
df_geolocation.duplicated(subset=['geolocation_zip_code_prefix']).sum()

np.int64(981148)

In [29]:
df_geolocation.duplicated().sum()

np.int64(261831)

In [30]:
df_geolocation.describe(include='object')

,geolocation_city,geolocation_state
count,1000163,1000163
unique,8011,27
top,sao paulo,SP
freq,135800,404268


### Initial Observations
- the table stores geographic coordinates for ZIP code prefixes used by customers and sellers
- each row represents a geographic location associated with a ZIP code prefix
- the table does not have a defined Primary Key or Foreign Key
- joining directly on geolocation_zip_code_prefix may duplicate customer records because a ZIP code prefix can have multiple geographic coordinate records
- 


## payment table

In [31]:
df_payments.sample(5)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
4746,aeb70ca14a984230b34a6cbf48fe6d8a,1,credit_card,8,184.10
71256,8eda634e0a441cc94d73885ca16ecadd,1,credit_card,5,310.28
65415,5999d0076050549364f1df892e1aeaa5,1,credit_card,1,43.82
38700,e267e6d5b54bdbcd941f9c8bc4eadb82,1,debit_card,1,267.76
76858,7ca1ad2143cdd9d02d4d9c1fe9656f4c,1,credit_card,10,233.10


In [32]:
df_payments.info()

print("\n\ncount of null values")
df_payments.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103886 entries, 0 to 103885
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   order_id              103886 non-null  object 
 1   payment_sequential    103886 non-null  int64  
 2   payment_type          103886 non-null  object 
 3   payment_installments  103886 non-null  int64  
 4   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 4.0+ MB


count of null values


order_id                0
payment_sequential      0
payment_type            0
payment_installments    0
payment_value           0
dtype: int64

In [33]:
payment_rows = df_payments.duplicated().sum()
# don't exist dup rows
order_id_dup = df_payments.duplicated(subset=['order_id']).sum()
composite = df_payments.duplicated(subset=['order_id', 'payment_sequential']).sum()

print(f"payment rows duplicated: {payment_rows}\n")
print(f"order_id_duplicated: {order_id_dup}\n")
print(f"composite order_id and payment_sequential duplicated: {composite}")


payment rows duplicated: 0

order_id_duplicated: 4446

composite order_id and payment_sequential duplicated: 0


In [34]:
df_payments.describe(include='object')

,order_id,payment_type
count,103886,103886
unique,99440,5
top,fa65dad1b0e818e3ccc5cb0e39231352,credit_card
freq,29,76795


In [35]:
df_payments[['payment_sequential','payment_installments','payment_value']].agg(['min', 'max'])
# some paymets have 0 value


,payment_sequential,payment_installments,payment_value
min,1,0,0.00
max,29,24,13664.08


In [36]:
# chack order_id with 29 payment_sequential
df_payments[df_payments.order_id == 'fa65dad1b0e818e3ccc5cb0e39231352'].sort_values(by= 'payment_sequential')

,order_id,payment_sequential,payment_type,payment_installments,payment_value
14321,fa65dad1b0e818e3ccc5cb0e39231352,1,voucher,1,3.71
23074,fa65dad1b0e818e3ccc5cb0e39231352,2,voucher,1,8.51
65641,fa65dad1b0e818e3ccc5cb0e39231352,3,voucher,1,2.95
9985,fa65dad1b0e818e3ccc5cb0e39231352,4,voucher,1,29.16
28330,fa65dad1b0e818e3ccc5cb0e39231352,5,voucher,1,0.66
29648,fa65dad1b0e818e3ccc5cb0e39231352,6,voucher,1,5.02
82593,fa65dad1b0e818e3ccc5cb0e39231352,7,voucher,1,0.32
68853,fa65dad1b0e818e3ccc5cb0e39231352,8,voucher,1,26.02
17274,fa65dad1b0e818e3ccc5cb0e39231352,9,voucher,1,1.08
19565,fa65dad1b0e818e3ccc5cb0e39231352,10,voucher,1,12.86


### Initial Observations
- the table stores payment information for customer orders
- each row represents one payment transaction associated with an order
- whit composite (order_id, payment_sequential) we have primary key
- an order may have multiple payment records, resulting in duplicate order_id values


## reviews table

In [37]:
df_reviews.sample(5)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
88537,5430589748eb1d2c1e48e65ce612fddd,5f321b4a49def66a561530fce7c03bab,1,NaN,gostaria de receber meu dinheiro de volta,2018-03-07 00:00:00,2018-03-07 10:16:50
27581,dadb368f1fdc539ad6381846141298f8,0eb7aec5fc39bddda0c24ee5cf824415,4,NaN,NaN,2018-04-18 00:00:00,2018-04-19 16:20:29
86168,e29ada46c54b907e0dc68d3394f1054a,a9687a65efa981b3d1e10a8b437962da,5,NaN,NaN,2017-11-29 00:00:00,2017-12-02 19:40:14
19732,b190f17d5b331ec4eb680c7edfefcf91,f64a76cad36b9c69a34699f1858db432,5,NaN,NaN,2018-02-01 00:00:00,2018-02-01 20:51:55
8795,3a5940370de5d5165fa0ef2f7fc3fb62,38ac0cc1879cabf9cb23a2e8f217acfe,5,NaN,NaN,2018-02-17 00:00:00,2018-02-17 21:36:09


In [38]:
df_reviews.info()

print("\n\ncount of null values")
df_reviews.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   review_id                99224 non-null  object
 1   order_id                 99224 non-null  object
 2   review_score             99224 non-null  int64 
 3   review_comment_title     11568 non-null  object
 4   review_comment_message   40977 non-null  object
 5   review_creation_date     99224 non-null  object
 6   review_answer_timestamp  99224 non-null  object
dtypes: int64(1), object(6)
memory usage: 5.3+ MB


count of null values


review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656
review_comment_message     58247
review_creation_date           0
review_answer_timestamp        0
dtype: int64

In [39]:
reviews_duplicated = df_reviews.duplicated().sum()
# don't exist dup rows

review_id = df_reviews.duplicated(subset=['review_id']).sum()
order_id = df_reviews.duplicated(subset=['order_id']).sum()
composite_reviews = df_reviews.duplicated(subset=['review_id', 'order_id']).sum()

#check to rows for null values
null_rows = df_reviews[['review_id', 'order_id']].isnull().sum()

print(f"reviews rows duplicated: {reviews_duplicated}\n")
print(f"review_id duplicated: {review_id}\n")
print(f"order_id duplicated: {order_id}\n")
print(f"composite_reviews duplicated: {composite_reviews}\n")
print(f"count of null values: \n{null_rows}")

reviews rows duplicated: 0

review_id duplicated: 814

order_id duplicated: 551

composite_reviews duplicated: 0

count of null values: 
review_id    0
order_id     0
dtype: int64


In [40]:
df_reviews.describe(include='object')

,review_id,order_id,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
count,99224,99224,11568,40977,99224,99224
unique,98410,98673,4527,36159,636,98248
top,7b606b0d57b078384f0b58eac1d41d78,c88b1d1b157a9999ce368f218a407141,Recomendo,Muito bom,2017-12-19 00:00:00,2017-06-15 23:21:05
freq,3,3,423,230,463,4


In [41]:
df_reviews.describe()

,review_score
count,99224.000000
mean,4.086421
std,1.347579
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


### Initial Observations
- the table stores reviews information for customer orders
- eachrow represents one review record associated with an order
- the combination of review_id and order_id is a candidate composite primary key
- some orders are associated with multiple review records, resulting in duplicate order_id values

## translation table

In [42]:
df_translation.sample(5)

,product_category_name,product_category_name_english
55,fashion_roupa_feminina,fashio_female_clothing
43,construcao_ferramentas_iluminacao,construction_tools_lights
0,beleza_saude,health_beauty
39,livros_tecnicos,books_technical
3,cama_mesa_banho,bed_bath_table


In [43]:
df_translation.info()

print("\n\ncount of null values")
df_translation.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   product_category_name          71 non-null     object
 1   product_category_name_english  71 non-null     object
dtypes: object(2)
memory usage: 1.2+ KB


count of null values


product_category_name            0
product_category_name_english    0
dtype: int64

In [44]:
df_translation.duplicated().sum()


np.int64(0)

In [45]:
df_translation.describe(include='object')

,product_category_name,product_category_name_english
count,71,71
unique,71,71
top,beleza_saude,health_beauty
freq,1,1


### Initial Observations

- each row represents one product category and its English translation
- `product_category_name` appears to uniquely identify each category and is a candidate Primary Key
- the table does not contain any Foreign Key constraints
- no duplicate records were found
- no missing values were found
